# Multi-Agent Travel Recommender

**Práctica de Big Data · Universitat Jaume I**

Este notebook desarrolla un sistema de recomendación de itinerarios turísticos con **dos agentes especializados** y un conjunto de datos de lugares turísticos.

- **Travel Data Analyst:** consulta y filtra un DataFrame con Pandas teniendo en cuenta ciudad, intereses y presupuesto.
- **Travel Planner:** organiza los lugares seleccionados en un itinerario diario y devuelve una salida estructurada.

El flujo separa el análisis de datos de la planificación. Al final se incluye una vista visual para presentar el resultado del workflow.

> **Antes de ejecutar:** añade `travel_places_dataset.csv` al entorno de Colab y configura las credenciales necesarias para el modelo. No publiques tus claves API en el notebook.

## 1. Carga del dataset

Leemos el archivo CSV con **Pandas** y lo convertimos en un DataFrame para trabajar con los destinos turísticos. El CSV debe estar disponible en el directorio de ejecución.

In [23]:
import pandas as pd

df = pd.read_csv("travel_places_dataset.csv")
df.head()

,city,place,category,zone,price_eur,duration_hours,rating,indoor,best_time,crowd_level,interest_tags,budget_level
0,Rome,Colosseum,History,Centro Storico,18,2.00,4.8,False,morning,high,history;architecture,medium
1,Rome,Roman Forum,History,Centro Storico,18,2.00,4.7,False,morning,high,history;archaeology,medium
2,Rome,Pantheon,History,Centro Storico,5,1.00,4.8,False,morning,high,history;architecture,low
3,Rome,Trevi Fountain,Landmark,Centro Storico,0,0.75,4.7,False,evening,high,architecture;photography,free
4,Rome,Piazza Navona,Landmark,Centro Storico,0,1.00,4.7,False,evening,medium,architecture;food;photography,free


### Exploración inicial

Comprobamos las dimensiones, los nombres de las columnas y las ciudades disponibles. Así verificamos que el dataset tiene la estructura esperada antes de utilizarlo.

In [24]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nCities:")
print(df["city"].unique())

Shape: (60, 12)

Columns:
['city', 'place', 'category', 'zone', 'price_eur', 'duration_hours', 'rating', 'indoor', 'best_time', 'crowd_level', 'interest_tags', 'budget_level']

Cities:
['Rome' 'Paris' 'Barcelona' 'Lisbon' 'London']


### Resumen estadístico

`describe()` proporciona estadísticas descriptivas de las columnas numéricas del DataFrame.

In [25]:
df.describe()

,price_eur,duration_hours,rating
count,60.000000,60.000000,60.000000
mean,9.816667,1.737500,4.668333
std,9.995748,0.584809,0.106551
min,0.000000,0.750000,4.400000
25%,0.000000,1.500000,4.600000
50%,10.000000,1.500000,4.700000
75%,15.250000,2.000000,4.700000
max,35.000000,3.000000,4.800000


## 2. Solicitud del usuario

Definimos `TravelRequest` con **Pydantic** para especificar los datos de entrada: ciudad, número de días, intereses y presupuesto máximo por actividad. Esto ayuda a mantener un contrato de entrada claro entre componentes.

In [26]:
from pydantic import BaseModel, Field
from typing import List

class TravelRequest(BaseModel):
  city: str = Field(description="Destination city")
  days: int = Field(description="Number of travel days")
  interests: List[str] = Field(description="User interests")
  max_price_per_activity: float = Field(description="Maximum price allowed for each activity")

## 3. Herramienta de análisis con Pandas

La función `filter_places` filtra los lugares del dataset según los parámetros recibidos. El decorador `@tool` la pone a disposición del agente de análisis para que pueda consultar los datos.

### Dependencias de Agno

Instalamos la librería de agentes y el cliente necesario para el modelo utilizado.

In [27]:
!pip install agno==3.0.6 openai -q

In [28]:
from agno.tools import tool

@tool(
    name="filter_places",
    description="Filters tourist places according to city, interests and maximum budget."
)
def filter_places(
    city: str,
    interests: list[str],
    max_price_per_activity: float
) -> list[dict]:
  filtered = df[
      (df["city"].str.lower() == city.lower()) &
      (df["price_eur"] <= max_price_per_activity)
  ].copy()

  if interests:
    interests_lower = [interest.lower() for interest in interests]

    filtered = filtered[
        filtered["interest_tags"].apply(
            lambda tags: any(
                interest in tags.lower()
                for interest in interests_lower
            )
        )
    ]

  # Priorizar lugares compatibles con mejor valoración.
  filtered = filtered.sort_values(
      by="rating",
      ascending=False
  )

  # Devolver registros serializables como diccionarios.
  return filtered.to_dict(orient="records")


### Prueba de la herramienta

Probamos `filter_places` de forma aislada con un viaje a Roma para verificar su filtrado antes de integrarla en el agente.

In [29]:
result = filter_places.entrypoint(
    city="Rome",
    interests=["history", "food"],
    max_price_per_activity=20
)

result

[{'city': 'Rome',
  'place': 'Colosseum',
  'category': 'History',
  'zone': 'Centro Storico',
  'price_eur': 18,
  'duration_hours': 2.0,
  'rating': 4.8,
  'indoor': False,
  'best_time': 'morning',
  'crowd_level': 'high',
  'interest_tags': 'history;architecture',
  'budget_level': 'medium'},
 {'city': 'Rome',
  'place': 'Pantheon',
  'category': 'History',
  'zone': 'Centro Storico',
  'price_eur': 5,
  'duration_hours': 1.0,
  'rating': 4.8,
  'indoor': False,
  'best_time': 'morning',
  'crowd_level': 'high',
  'interest_tags': 'history;architecture',
  'budget_level': 'low'},
 {'city': 'Rome',
  'place': "St Peter's Basilica",
  'category': 'Religious',
  'zone': 'Vatican',
  'price_eur': 0,
  'duration_hours': 1.5,
  'rating': 4.8,
  'indoor': True,
  'best_time': 'morning',
  'crowd_level': 'high',
  'interest_tags': 'history;architecture',
  'budget_level': 'free'},
 {'city': 'Rome',
  'place': 'Galleria Borghese',
  'category': 'Museum',
  'zone': 'Borghese',
  'price_eur':

La herramienta devuelve registros del DataFrame compatibles con la ciudad, los intereses y el límite de precio. Los lugares se ordenan por valoración para facilitar su posterior selección.

## 4. Configuración del primer agente

Preparamos el entorno y definimos un agente que puede utilizar la herramienta de análisis para producir una selección de lugares.

### Configuración de la API de OpenAI

Para ejecutar este notebook es necesario disponer de una API Key de OpenAI.

1. Obtén una clave.
2. Ejecuta la siguiente celda.
3. Introduce tu clave cuando se solicite.
4. Continúa ejecutando el notebook normalmente.

La clave se mantiene en la sesión actual y no se muestra en las salidas del notebook. El uso de la API puede generar costes según el modelo utilizado.

In [30]:
import os
from getpass import getpass

# Introducir la clave de OpenAI para ejecutar el proyecto
os.environ["OPENAI_API_KEY"] = getpass("Introduce tu API Key de OpenAI: ")

print("Clave API configurada correctamente.")


Introduce tu API Key de OpenAI: ··········
Clave API configurada correctamente.


### Agente de análisis: versión inicial

El agente utiliza el modelo de lenguaje y la herramienta `filter_places` para seleccionar destinos. Más adelante se añadirá un esquema de salida estructurada.

In [31]:
from agno.agent import Agent
from agno.models.openai import OpenAIResponses

LLM_model_name = "gpt-5.6-luna"

travel_analyst = Agent(
    name="Travel Data Analyst",
    # Modelo de lenguaje empleado por el agente.
    model=OpenAIResponses(id=LLM_model_name),
    # Herramienta de consulta sobre el dataset.
    tools=[filter_places],
    # Esquema de entrada definido con Pydantic.
    input_schema=TravelRequest,
    instructions="""
      Eres un analista de viajes. Tu tarea consiste en analizar la solicitud
      de viaje del usuario y consulta la herramienta filter_places si es necesario,
      para encontrar lugares turísticos dentro del conjunto de datos.

      Utiliza los parámetros que llegan a la función (ciudad, intereses y presupuesto)
      para seleccionar lugares relevantes.

      Presenta los lugares seleccionados de forma clara para que otro agente pueda
      utilizarlos posteriormente en la creación de un itinerario de viaje.
    """
)

In [32]:
request = TravelRequest(
    city="Rome",
    days=2,
    interests=["history", "food"],
    max_price_per_activity=20
)

response = travel_analyst.run(request, debug_mode=True)
print(response.content)

DEBUG ************** Agent ID: travel-data-analyst ***************

DEBUG Creating new AgentSession: d292f6f6-1409-4d4d-92d4-1325ae00db14

DEBUG ** Agent Run Start: 800d6208-4ce9-43f1-a2fa-398d2e460796 ***

DEBUG Processing tools for model

DEBUG Added tool filter_places

...

DEBUG ************************  METRICS  *************************

DEBUG -------------- OpenAI Responses Response End ---------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 800d6208-4ce9-43f1-a2fa-398d2e460796 ****

Lugares turísticos seleccionados en **Roma** para 2 días, con intereses en **historia y gastronomía**, y un presupuesto máximo de **20 € por actividad**:

### Historia y patrimonio

| Lugar | Zona | Precio | Duración | Mejor momento | Valoración |
|---|---|---:|---:|---|---:|
| Coliseo | Centro Storico | 18 € | 2 h | Mañana | 4,8 |
| Foro Romano | Centro Storico | 18 € | 2 h | Mañana | 4,7 |
| Panteón | Centro Storico | 5 € | 1 h | Mañana | 4,8 |
| Basílica de San Pedro | Vaticano | Gratis | 1,5 h | Mañana | 4,8 |
| Castel Sant’Angelo | Prati | 16 € | 1,5 h | Tarde | 4,6 |
| Galería Borghese | Borghese | 15 € | 2 h | Tarde | 4,8 |

### Gastronomía y ambiente local

| Lugar | Zona | Precio orientativo | Duración | Mejor momento | Valoración |
|---|---|---:|---:|---|---:|
| Trastevere | Trastevere | 15 € | 2 h | Tarde-noche | 4,6 |
| Campo de’ Fiori | Centro Storico | 10 € | 1 h | Mañana | 4,4 |
| Piazza Navona | Centro Storico | Gratis | 1 h | Tarde-noche | 4,7 |

### Consideraciones pa

El primer agente consulta la herramienta y selecciona lugares según la petición. Para que el segundo agente pueda consumir su resultado de forma fiable, definimos a continuación una **salida estructurada con Pydantic**.

### Contrato de salida del analista

`TravelSelection` contiene la ciudad, el número de días y la lista de lugares seleccionados (`SelectedPlace`).

In [33]:
from pydantic import BaseModel
from typing import List

class SelectedPlace(BaseModel):
  place: str
  category: str
  zone: str
  price_eur: float
  duration_hours: float
  rating: float
  best_time: str
  interest_tags: str

class TravelSelection(BaseModel):
  city: str
  days: int
  selected_places: List[SelectedPlace]

### Agente de análisis: salida estructurada

Volvemos a definir el analista con `output_schema=TravelSelection` para normalizar el resultado que pasará al planificador.

In [34]:
#Modificamos el agente para definir la salida que deseamos
travel_analyst = Agent(
    name="Travel Data Analyst",
    # Modelo de lenguaje empleado por el agente.
    model=OpenAIResponses(id=LLM_model_name),
    # Herramienta de consulta sobre el dataset.
    tools=[filter_places],
    # Esquema de entrada definido con Pydantic.
    input_schema=TravelRequest,
    output_schema=TravelSelection,
    instructions="""
      Eres un analista de viajes. Tu tarea consiste en analizar la solicitud
      de viaje del usuario y consulta la herramienta filter_places si es necesario,
      para encontrar lugares turísticos dentro del conjunto de datos.

      Utiliza los parámetros que llegan a la función (ciudad, intereses y presupuesto)
      para seleccionar lugares relevantes.

      Devuelve el resultado usando la estructura TravelSelection.
    """,
    markdown=True,
    debug_mode=True
)

In [35]:
request = TravelRequest(
    city="Rome",
    days=2,
    interests=["history", "food"],
    max_price_per_activity=20
)

response = travel_analyst.run(request)

print(response.content)
print(type(response.content))

DEBUG ***** Session ID: 929d725b-a0ca-4f54-bb07-3e67368ea004 *****

DEBUG ************** Agent ID: travel-data-analyst ***************

DEBUG Creating new AgentSession: 929d725b-a0ca-4f54-bb07-3e67368ea004

DEBUG Setting Model.response_format to Agent.output_schema

DEBUG ** Agent Run Start: 0b82cc81-1a34-48c6-9cb1-233a15b77927 ***

...

DEBUG ************************  METRICS  *************************

DEBUG -------------- OpenAI Responses Response End ---------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 0b82cc81-1a34-48c6-9cb1-233a15b77927 ****

city='Rome' days=2 selected_places=[SelectedPlace(place='Colosseum', category='History', zone='Centro Storico', price_eur=18.0, duration_hours=2.0, rating=4.8, best_time='morning', interest_tags='history;architecture'), SelectedPlace(place='Roman Forum', category='History', zone='Centro Storico', price_eur=18.0, duration_hours=2.0, rating=4.7, best_time='morning', interest_tags='history;archaeology'), SelectedPlace(place='Pantheon', category='History', zone='Centro Storico', price_eur=5.0, duration_hours=1.0, rating=4.8, best_time='morning', interest_tags='history;architecture'), SelectedPlace(place="Campo de' Fiori", category='Market', zone='Centro Storico', price_eur=10.0, duration_hours=1.0, rating=4.4, best_time='morning', interest_tags='food;local-life'), SelectedPlace(place='Piazza Navona', category='Landmark', zone='Centro Storico', price_eur=0.0, duration_hours=1.0, rating=4.7, best_time='evening', interest_tags='architecture;food;photography'), SelectedPlace(place='Trastevere'

## 5. Agente planificador

El segundo agente recibe una `TravelSelection` y organiza las actividades en días. Antes de definir el agente, creamos el esquema `TravelPlan`, con sus días, actividades, duraciones y costes.

In [36]:
from pydantic import BaseModel
from typing import List

class PlannedActivity(BaseModel):
    place: str
    time_of_day: str
    estimated_duration: float
    price_eur: float

class DayPlan(BaseModel):
    day: int
    activities: List[PlannedActivity]
    total_cost: float
    total_duration: float

class TravelPlan(BaseModel):
    city: str
    days: List[DayPlan]
    total_trip_cost: float

### Configuración del planificador

El agente de planificación recibe una selección de lugares y devuelve un `TravelPlan`. Se le indica que utilice únicamente los lugares proporcionados.

In [37]:
#Creamos el agente
#La salida del primer agente encaja como la entrada del segundo
travel_planner = Agent(
    name="Travel Planner",
    model=OpenAIResponses(id=LLM_model_name),
    input_schema=TravelSelection,
    output_schema=TravelPlan,
    instructions="""
    Eres un planificador de viajes.

    Tu tarea consiste en crear un itinerario de viaje personalizado
    utilizando los lugares turísticos seleccionados por el agente
    anterior.

    Organiza los lugares seleccionados a lo largo de los días disponibles.

    Ten en cuenta: dar preferencia a lugares mejor valorados,
    intenta elegir actividades en el momento adecuado del día
    y sin sobrecargar la agenda diaria con actividades.

    Usa únicamente los lugares proporcionados en la información
    de entrada. No inventes lugares turísticos.

    Devuelve el itinerario utilizando la estructura TravelPlan.
    """,
    markdown=True,
    debug_mode=True
)

## 6. Workflow multiagente

Encadenamos analista y planificador y ejecutamos una solicitud de ejemplo a Roma. El workflow necesita acceso al servicio del modelo para completarse.

In [38]:
from agno.workflow import Workflow

travel_workflow = Workflow(
    name="Multi-Agent Travel Planner",
    steps=[
        travel_analyst,
        travel_planner
    ]
)

In [39]:
request = TravelRequest(
    city="Rome",
    days=2,
    interests=["history", "food"],
    max_price_per_activity=20
)

response = travel_workflow.run(request)

print(response.content)

print("Tipo de respuesta:", type(response.content).__name__)

DEBUG ************** Agent ID: travel-data-analyst ***************

DEBUG Creating new AgentSession: 124b833f-8021-4525-867a-fbad35aff2c2

DEBUG Setting Model.response_format to Agent.output_schema

DEBUG ** Agent Run Start: 868ea963-ffc3-477a-bf23-3cc676632437 ***

DEBUG Processing tools for model

...

DEBUG ************************  METRICS  *************************

DEBUG -------------- OpenAI Responses Response End ---------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 471b9a9e-68f2-4cd0-81e7-00b787a9f23a ****

city='Rome' days=[DayPlan(day=1, activities=[PlannedActivity(place='Colosseum', time_of_day='morning', estimated_duration=2.0, price_eur=18.0), PlannedActivity(place='Roman Forum', time_of_day='morning', estimated_duration=2.0, price_eur=18.0), PlannedActivity(place="Campo de' Fiori", time_of_day='late morning', estimated_duration=1.0, price_eur=10.0), PlannedActivity(place='Pantheon', time_of_day='afternoon', estimated_duration=1.0, price_eur=5.0)], total_cost=51.0, total_duration=6.0), DayPlan(day=2, activities=[PlannedActivity(place="St Peter's Basilica", time_of_day='morning', estimated_duration=1.5, price_eur=0.0), PlannedActivity(place="Castel Sant'Angelo", time_of_day='afternoon', estimated_duration=1.5, price_eur=16.0), PlannedActivity(place='Piazza Navona', time_of_day='evening', estimated_duration=1.0, price_eur=0.0), PlannedActivity(place='Trastevere', time_of_day='evening', estimated_duration=2.0, price_eur=15.0)], total_cost=31.0, total_duration=6.0)] total_trip_cost=82.0


### Inspección del workflow

`step_results` permite revisar la ejecución de cada etapa y diagnosticar posibles errores de los agentes.

## 7. Conclusiones

El prototipo organiza el problema en **dos etapas con responsabilidades diferenciadas**: el primer agente filtra lugares turísticos apoyándose en un DataFrame de Pandas, mientras que el segundo construye el itinerario con los lugares seleccionados.

Esta separación permite estudiar el intercambio de información estructurada entre agentes y facilita la ampliación del sistema con nuevas reglas de selección o planificación.

## 8. Presentación del itinerario

La última celda convierte el resultado de la ejecución anterior en una vista HTML apta para una captura de pantalla. **No llama de nuevo al modelo ni genera itinerarios ficticios**. Si la respuesta no tiene el formato de `TravelPlan`, muestra una advertencia y el contenido disponible para que pueda diagnosticarse.

In [40]:
import json
import re
from html import escape
from IPython.display import HTML, display
from pydantic import BaseModel


def as_dict(value):
    """Convierte una salida estructurada a dict; si es texto, intenta JSON válido."""
    if isinstance(value, BaseModel):
        return value.model_dump()
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        text = value.strip()
        # A veces la respuesta viene envuelta en un bloque Markdown.
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
        text = re.sub(r"\s*```$", "", text)
        try:
            parsed = json.loads(text)
            return parsed if isinstance(parsed, dict) else None
        except (json.JSONDecodeError, TypeError):
            return None
    return None


def render_travel_report(response, request):
    """Muestra el plan real si es estructurado; de lo contrario ofrece diagnóstico."""
    content = getattr(response, "content", response)
    plan = as_dict(content)

    css = """
    <style>
    .tr-report {font-family:Arial,sans-serif;background:#0b1120;color:#e5e7eb;
      padding:28px;border-radius:15px;max-width:940px;margin:8px auto;}
    .tr-report h1 {color:#2dd4bf;margin:0 0 6px;font-size:26px;}
    .tr-report h2 {font-size:19px;margin:25px 0 13px;color:#f8fafc;}
    .tr-report h3 {font-size:17px;color:#2dd4bf;margin:0 0 8px;}
    .tr-muted {color:#94a3b8;font-size:14px;line-height:1.5;}
    .tr-grid {display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px;margin-top:20px;}
    .tr-card {background:#111c2d;border:1px solid #334155;border-radius:10px;padding:16px;min-width:0;}
    .tr-label {color:#94a3b8;font-size:11px;letter-spacing:.06em;text-transform:uppercase;margin-bottom:7px;}
    .tr-value {font-weight:650;overflow-wrap:anywhere;}
    .tr-day {margin-bottom:14px;}
    .tr-activity,.tr-total-line {display:flex;justify-content:space-between;align-items:start;
      gap:16px;padding:10px 0;border-bottom:1px solid #263449;}
    .tr-activity:last-of-type {border-bottom:0;}
    .tr-price {white-space:nowrap;}
    .tr-total {display:flex;justify-content:space-between;gap:16px;background:#134e4a;
      padding:18px;border-radius:10px;font-weight:bold;font-size:17px;}
    .tr-warn {background:#3b2d17;border:1px solid #a16207;padding:17px;border-radius:10px;margin-top:16px;}
    .tr-raw {white-space:pre-wrap;overflow-wrap:anywhere;font-family:Consolas,monospace;
      font-size:12px;background:#111c2d;border-radius:8px;padding:15px;max-height:330px;overflow:auto;}
    @media(max-width:540px){.tr-grid{grid-template-columns:1fr}.tr-report{padding:17px}}
    </style>
    """
    title = '<h1>Multi-Agent Travel Recommender</h1><p class="tr-muted">Itinerario generado mediante dos agentes</p>'

    city = escape(str(getattr(request, "city", "Sin especificar")))
    interests = escape(", ".join(map(str, getattr(request, "interests", []))))
    days_requested = escape(str(getattr(request, "days", "—")))
    max_price = getattr(request, "max_price_per_activity", None)

    def euro(value):
        try:
            return f"{float(value):.2f} €"
        except (ValueError, TypeError):
            return "No disponible"

    summary = f"""
      <div class="tr-grid">
        <div class="tr-card"><div class="tr-label">Destino</div><div class="tr-value">{city}</div></div>
        <div class="tr-card"><div class="tr-label">Duración</div><div class="tr-value">{days_requested} días</div></div>
        <div class="tr-card"><div class="tr-label">Intereses</div><div class="tr-value">{interests or 'Sin preferencias'}</div></div>
        <div class="tr-card"><div class="tr-label">Máximo por actividad</div><div class="tr-value">{euro(max_price)}</div></div>
      </div>
    """
    # Acepta únicamente los campos esperados en el esquema TravelPlan.
    if not isinstance(plan, dict) or not isinstance(plan.get("days"), list):
        raw_text = str(content) if content is not None else "Respuesta vacía."
        warning = (
            '<div class="tr-warn"><strong>El workflow no devolvió un TravelPlan estructurado.</strong>'
            '<p class="tr-muted">Se muestra la respuesta real para revisar el resultado. '
            'Comprueba también response.step_results y la configuración de los agentes.</p></div>'
            f'<h2>Respuesta recibida</h2><div class="tr-raw">{escape(raw_text[:12000])}</div>'
        )
        display(HTML(css + '<div class="tr-report">' + title + summary + warning + '</div>'))
        return False

    cards = []
    for i, day in enumerate(plan["days"], 1):
        if not isinstance(day, dict):
            continue
        activities = []
        for activity in day.get("activities", []):
            if not isinstance(activity, dict):
                continue
            place = escape(str(activity.get("place", "Actividad sin nombre")))
            time = escape(str(activity.get("time_of_day", "Horario no especificado")))
            duration = escape(str(activity.get("estimated_duration", "—")))
            price = euro(activity.get("price_eur"))
            activities.append(f"""<div class="tr-activity"><div><strong>{place}</strong>
              <div class="tr-muted">{time} · {duration} h</div></div>
              <span class="tr-price">{price}</span></div>""")
        day_num = escape(str(day.get("day", i)))
        subtotal = euro(day.get("total_cost"))
        cards.append(f"""<div class="tr-card tr-day"><h3>Día {day_num}</h3>
        {''.join(activities) if activities else '<p class="tr-muted">Sin actividades.</p>'}
        <div class="tr-total-line"><span>Coste del día</span><strong>{subtotal}</strong></div></div>""")

    total = euro(plan.get("total_trip_cost"))
    body = f"""<h2>Itinerario recomendado</h2>{''.join(cards) or '<p>Sin días disponibles.</p>'}
      <div class="tr-total"><span>Coste total del itinerario</span><span>{total}</span></div>"""
    display(HTML(css + '<div class="tr-report">' + title + summary + body + '</div>'))
    return True


# Usa `response` y `request` generados por la celda anterior.
if "response" not in globals() or "request" not in globals():
    print("Ejecuta primero la celda del workflow para crear 'request' y 'response'.")
else:
    render_travel_report(response, request)
